In [2]:
import numpy as np 
import torch.optim as optim    
from tqdm import tqdm 

def train(model, device, trainloader, optimizer, criterion, num_epochs):
    history = np.zeros((0, 3))  

    for epoch in tqdm(range(num_epochs)):
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0

        for X, y in trainloader: 
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            predict = model(X)
            loss = criterion(predict, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            pred_class = predict.argmax(dim=1)
            correct += (pred_class == y).sum().item()
            total += y.size(0)

        avg_loss = epoch_loss / len(trainloader)
        avg_accuracy = correct / total

        item = np.array([epoch + 1, avg_loss, avg_accuracy])
        history = np.vstack((history, item))

        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.6f}, Accuracy: {avg_accuracy:.4f}")

    return history
    
def test(model, device, test_loader, criterion):
    test_loss = []
    test_accuracy = []
    model.eval()

    with torch.no_grad():
        for X, y in tqdm(test_loader): 
            X = X.to(device)
            y = y.to(device)

            predict = model(X)
            loss = criterion(predict, y)

            pred_class = predict.argmax(dim=1)
            accuracy = (pred_class == y).float().mean()

            test_accuracy.append(accuracy.item())
            test_loss.append(loss.item())

    avg_loss = sum(test_loss) / len(test_loss)
    avg_accuracy = sum(test_accuracy) / len(test_accuracy)
    print(f'test loss : {avg_loss:.4f} / test_accuracy : {avg_accuracy:.4f}')

In [1]:
## Dataset ## 
import os
import torch
from torchvision import datasets, transforms

size_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),  
])

trainset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Train',  transform = size_transform) 
testset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Test',  transform = size_transform) 
trainloader = torch.utils.data.DataLoader(trainset, batch_size=20, shuffle=True, num_workers=2, drop_last=True)
testloader = torch.utils.data.DataLoader(testset, batch_size = 20, shuffle=True, num_workers=2, drop_last=True)

In [3]:
## resnet model ##
import torchvision.models as models
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
prob1_1 = models.resnet18()
num_ftrs = prob1_1.fc.in_features
prob1_1.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
prob1_1.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4),
)

prob1_1 = prob1_1.to(device)

In [13]:
# 하이퍼파라미터 설정
num_epochs = 40
lr = 0.001
optimizer = torch.optim.Adam(prob1_1.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()

# 학습 실행
history = train(prob1_1, device, trainloader, optimizer, criterion, num_epochs)

  2%|█▊                                                                      | 1/40 [00:50<32:38, 50.22s/it]

Epoch [1/40] - Loss: 0.841812, Accuracy: 0.6411


  5%|███▌                                                                    | 2/40 [01:40<31:53, 50.34s/it]

Epoch [2/40] - Loss: 0.688757, Accuracy: 0.7280


  8%|█████▍                                                                  | 3/40 [02:31<31:05, 50.42s/it]

...

Epoch [38/40] - Loss: 0.036581, Accuracy: 0.9882


 98%|█████████████████████████████████████████████████████████████████████▏ | 39/40 [32:57<00:50, 50.78s/it]

Epoch [39/40] - Loss: 0.017469, Accuracy: 0.9946


100%|███████████████████████████████████████████████████████████████████████| 40/40 [33:48<00:00, 50.72s/it]

Epoch [40/40] - Loss: 0.004128, Accuracy: 0.9988


In [21]:
result = test(prob1_1, device, testloader, criterion)

100%|███████████████████████████████████████████████████████████████████████| 39/39 [00:01<00:00, 21.97it/s]

test loss : 0.7112 / test_accuracy : 0.8795
